Pyspark tutorial : https://www.kaggle.com/code/nilaychauhan/pyspark-tutorial-for-beginners

In [9]:
import os

In [10]:
DIR = r'/kaggle/input/playground-series-s6e1'
TEST = os.path.join(DIR, r'test.csv')
TRAIN = os.path.join(DIR, r'train.csv')

In [2]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

Create a **SparkContext** object. With the SparkContext, you can input a dataset and parallelize the data across a cluster.

In [5]:
spark = SparkSession \
    .builder \
    .appName("Pred_Student_Scores_EDA") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/22 07:35:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
spark.sparkContext.getConf().getAll()

[('spark.driver.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false'),
 ('spark.executor.id', 'driver'),
 ('spark.app.startTime', '1769067323528'),
 ('spark.app.id', 'local-1

In [7]:
spark

### Reading the Dataframe using spark

By default, Spark assumes there is no header and all columns are of type StringType, named _c0, _c1, etc.. 

### Common Options
You can customize the reading behavior using options for more robust data loading: 

Option | Default | Description | Example |
-------| ------- | ----------- | ------- |
header	| `False`	| Set to `True` if the first row is the column header. | `header=True` |
inferSchema	| `False` | Set to `True` to automatically determine column data types (involves an extra pass over the data, which can be slow for large files).	| `inferSchema=True` |
sep	| `,`	| Specifies the column delimiter if it's not a comma (e.g., `'\\t'` for tabs, \`'	'\` for pipes). | |
mode | `PERMISSIVE` | Controls handling of corrupt records: `PERMISSIVE` (default), `DROPMALFORMED` (drops rows with corrupt data), or `FAILFAST` (aborts the task).	| `mode="DROPMALFORMED"` |

In [11]:
# Reading the csv file into a dataframe
df = spark.read.csv(TEST, header=True, inferSchema=True)

# Alternate code:
# df_options_alt = (
#     spark.read.format("csv")
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .load("path/to/yourfile.csv")
# )

#### Reading Multiple Files
From a single directory: Pass the path to the directory, and PySpark will read all CSV files within it.
```python
df_dir = spark.read.csv("path/to/directory/")
```
From multiple specific paths: Pass a list of file paths.
```python
paths = ["path/to/file1.csv", "path/to/file2.csv"]
df_multiple = spark.read.csv(paths, header=True, inferSchema=True)
```

#### Custom Schema in Pyspark

In PySpark, you specify a custom schema using the `StructType` class, which is a collection of `StructField` objects. This allows you to explicitly define column names, data types, and nullability, rather than relying on automatic schema inference. 

##### Steps to Define a Custom Schema
1. **Import necessary types:** Import `StructType`, `StructField`, and the specific data types you need (e.g., `StringType`, `IntegerType`, `DateType`) from `pyspark.sql.types`.
2. **Define the schema:** Create an instance of `StructType` containing a list of `StructField` instances. Each `StructField` takes the column name, data type, and a boolean indicating nullability (whether the column can contain `None` values).
3. **Apply the schema:**
    * When creating a DataFrame from a local collection, pass the schema to the `spark.createDataFrame()` method.
    * When reading data from a file (like CSV or JSON), pass the schema to the reader method (e.g., `.schema()`).


#### Example: Defining and Applying a Schema
This example demonstrates how to define a custom schema and use it when creating a DataFrame from a list of data. 
For an example of defining a schema and applying it when creating a DataFrame from data, you can refer to medium.com. 

**Example: Applying Schema When Reading a File**
A custom schema can be applied when reading data from files like CSV using the `.schema()` option, which offers better performance than schema inference. 

```python
# Assuming you have a custom_schema defined
df_csv = (spark.read
    .format("csv")
    .option("header", True) # Assuming your CSV has a header
    .schema(custom_schema) # Specify the custom schema
    .load("path/to/your/data.csv")
)

df_csv.printSchema()

```

**Complex and Nested Schemas**
For complex data structures such as nested JSON, a schema can be defined by embedding `StructType` within a `StructField`. An example of a nested schema definition is available at [medium.com](https://medium.com/@softwareprocesspains2023/pyspark-how-to-define-a-custom-nested-schema-for-a-dataframe-and-how-its-displayed-in-a-hive-1c054f632ff4).


Also refer to [stockoverflow.com](https://stackoverflow.com/questions/57901493/pyspark-defining-custom-schema-for-a-dataframe#:~:text=Related,schema%20of%20a%20df%20pyspark).


In [12]:
# Displaying the dataframe
df.show()

+------+---+------+-------+-----------+----------------+---------------+-----------+-------------+-------------+---------------+---------------+
|   _c0|_c1|   _c2|    _c3|        _c4|             _c5|            _c6|        _c7|          _c8|          _c9|           _c10|           _c11|
+------+---+------+-------+-----------+----------------+---------------+-----------+-------------+-------------+---------------+---------------+
|    id|age|gender| course|study_hours|class_attendance|internet_access|sleep_hours|sleep_quality| study_method|facility_rating|exam_difficulty|
|630000| 24| other|     ba|       6.85|            65.2|            yes|        5.2|         poor|  group study|           high|           easy|
|630001| 18|  male|diploma|       6.61|            45.0|             no|        9.3|         poor|     coaching|            low|           easy|
|630002| 24|female| b.tech|        6.6|            98.5|            yes|        6.2|         good|  group study|         medium|  

In [13]:
# Displaying the dataframe's schema
df.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)



# Exploratory Data Analysis